_Info: Some of the codes are inspired from the skin_cancer_classification.ipynb_

# Short data description
### ISIC 2016:

The International Skin Imaging Collaboration (ISIC) is an academia and industry partnership focused on digital skin imaging for melanoma detection.
The ISIC archive contains over 150,000 clinical and dermoscopic images with metadata, including diagnosis, anatomical location, size, melanoma features, patient information, and segmentation masks.

### PH2:
The PH2 dataset is a collection of skin images and masks that highlight specific areas that are interesting for medical purposes. 
There are also classification files (.xlsx, .txt) that describe the images based on certain skin characteristics. 
One can use this dataset to study and analyze skin lesions and other conditions.

# Overview
In the assignment i will examine both datasets and see if there are any unusual things in the data, like mistakes or biases. I will also check the distribution for each type of diagnosis and i will look at the shapes and sizes of the images to see if there are any special patterns in the data. I will figure out if i can use both datasets together for a specific job. Once that's done, i will divide the PH2 dataset into two parts: one for training and another for testing how good it is. Lastly, i will load the PH2 dataset using keras.

# Import Libraries

In [82]:
import os
import numpy as np
import pandas as pd
import shutil

import plotly.express as px

from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from pathlib import Path

# Data loading

In [84]:
DATA_ISIC = 'data/ISBI2016_ISIC_Part3B'
IMAGE_ISIC_TRAIN = 'data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Training_Data'
IMAGE_ISIC_TEST = 'data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data'

DATA_PH2 = 'data/PH2Dataset'
IMAGE_PH2 = 'data/PH2Dataset/PH2 Dataset images'
CLASS_NAMES = ['benign', 'malignant']
CLASS_NAMES = ['Atypical Nevus', 'Common Nevus', 'Melanoma']

In [85]:
lab_df = pd.read_csv(f'{DATA_ISIC}/ISBI2016_ISIC_Part3B_Training_GroundTruth.csv', names=['imageID', 'diagnosis'], header=None)
print(f'Shape of ISBIISBI2016_ISIC_Part3B: {lab_df.shape}')
df_ph2 = pd.read_excel(os.path.join(DATA_PH2, 'PH2_dataset.xlsx'), header=12)
print(f"Shpae of PH2 dataset: {df_ph2.shape}")
df_ph2.head(2)

def load_data(csv_file):
  lab_df = pd.read_csv(csv_file, names=['imageID', 'diagnosis'], header=None)
  lab_df.diagnosis.replace({i: c for i, c in enumerate(CLASS_NAMES)}, inplace=True)
  return lab_df

lab_train = load_data(f'{DATA_ISIC}/ISBI2016_ISIC_Part3B_Training_GroundTruth.csv')
lab_test = load_data(f'{DATA_ISIC}/ISBI2016_ISIC_Part3B_Test_GroundTruth.csv')

Shape of ISBIISBI2016_ISIC_Part3B: (900, 2)
Shpae of PH2 dataset: (200, 17)


In [86]:
df_ph2.head()

,Image Name,Histological Diagnosis,Common Nevus,Atypical Nevus,Melanoma,Asymmetry\n(0/1/2),Pigment Network\n(AT/T),Dots/Globules\n(A/AT/T),Streaks\n(A/P),Regression Areas\n(A/P),Blue-Whitish Veil\n(A/P),White,Red,Light-Brown,Dark-Brown,Blue-Gray,Black
0,IMD003,NaN,X,NaN,NaN,0,T,A,A,A,A,NaN,NaN,NaN,X,NaN,NaN
1,IMD009,NaN,X,NaN,NaN,0,T,A,A,A,A,NaN,NaN,X,NaN,NaN,NaN
2,IMD016,NaN,X,NaN,NaN,0,T,T,A,A,A,NaN,NaN,X,X,NaN,NaN
3,IMD022,NaN,X,NaN,NaN,0,T,A,A,A,A,NaN,NaN,X,NaN,NaN,NaN
4,IMD024,NaN,X,NaN,NaN,0,T,A,A,A,A,NaN,NaN,X,X,NaN,NaN


The clinical diagnosis (Common Nevus, Atypical Nevus, Melanoma) and the colors (White, Red, Light-Brown, Dark-Brown, Blue-Gray, Black) are Marked with *X* if chosen and *NaN* if not. Mapping them to 0 and 1 respectively is done in the next cell.

In [87]:
df_ph2 = df_ph2.fillna(0).replace({'X': 1})
df_ph2.head()

,Image Name,Histological Diagnosis,Common Nevus,Atypical Nevus,Melanoma,Asymmetry\n(0/1/2),Pigment Network\n(AT/T),Dots/Globules\n(A/AT/T),Streaks\n(A/P),Regression Areas\n(A/P),Blue-Whitish Veil\n(A/P),White,Red,Light-Brown,Dark-Brown,Blue-Gray,Black
0,IMD003,0,1,0,0,0,T,A,A,A,A,0,0,0,1,0,0
1,IMD009,0,1,0,0,0,T,A,A,A,A,0,0,1,0,0,0
2,IMD016,0,1,0,0,0,T,T,A,A,A,0,0,1,1,0,0
3,IMD022,0,1,0,0,0,T,A,A,A,A,0,0,1,0,0,0
4,IMD024,0,1,0,0,0,T,A,A,A,A,0,0,1,1,0,0


Now i will loop through each column and count unique values occurence. To get a better overview of the PH2 dataset

In [88]:
column_data_types = df_ph2.dtypes
for column_name, data_type in column_data_types.items():
    unique_values = df_ph2[column_name].value_counts(dropna=False)
    print(f"\nColumn: '{column_name}' (dtype: {data_type}) has {len(unique_values)} unique values:")
    print(unique_values)


Column: 'Image Name' (dtype: object) has 200 unique values:
Image Name
IMD003    1
IMD312    1
IMD243    1
IMD251    1
IMD254    1
         ..
IMD384    1
IMD385    1
IMD389    1
IMD390    1
IMD435    1
Name: count, Length: 200, dtype: int64

Column: 'Histological Diagnosis' (dtype: object) has 7 unique values:
Histological Diagnosis
0                    159
Melanoma              26
Dysplastic Nevus       5
Lentigo Maligna        5
Intradermal Nevus      2
Nodular Melanoma       2
Blue Nevus             1
Name: count, dtype: int64

Column: 'Common Nevus' (dtype: int64) has 2 unique values:
Common Nevus
0    120
1     80
Name: count, dtype: int64

Column: 'Atypical Nevus' (dtype: int64) has 2 unique values:
Atypical Nevus
0    120
1     80
Name: count, dtype: int64

Column: 'Melanoma' (dtype: int64) has 2 unique values:
Melanoma
0    160
1     40
Name: count, dtype: int64

Column: 'Asymmetry
(0/1/2)' (dtype: int64) has 3 unique values:
Asymmetry\n(0/1/2)
0    117
2     52
1     31
Name

# `Task 1: Explore and evaluate both datasets`

## Biasing artifacts

I have viewd at least 20-30 random samples from the PH2 dataset images and i can see that the images are dermoscopy images and all have good lightning with very less reflection. If hairfolicles are present or dry skin surface, than the pictures are also focused on that part which leads to some small biases. All have a small black border around the edges but not that significant or which would lead to bad resolution of the image. The lesion on the skin are very good porportioned to the image and there are no blurred images. I also see no clinical markings or bandages.

The ISIC dataset has several biases. Only some of them are displayed here:

| Images: isic10                     | Description                |
|-----------------------------------|-----------------------------|
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0000036.jpg" width="200" height="150"> |  Border   |
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0000043.jpg" width="200" height="150"> |  Hair   |
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0000231.jpg" width="200" height="150"> |  Ruler   |
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0000494.jpg" width="200" height="150"> |  Scale/Ruler   |
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0000299.jpg" width="200" height="150"> |  Zoom   |
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0000302.jpg" width="200" height="150"> |  Zoom   |
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0000328.jpg" width="200" height="150"> |  Lightning   |
| <img src="./data/ISBI2016_ISIC_Part3B/ISBI2016_ISIC_Part3B_Test_Data/ISIC_0002246.jpg" width="200" height="150"> |  Object   |

## Lable Distribution

The PH2 dataset consists of 200 dermoscopic images, categorized into three classes: 80 Common Nevus, 80 Atypical Nevus, and 40 Melanomas. The dataset appears to have an imbalanced class distribution with a smaller number of melanomas compared to the other classes.

The ISBI2016 dataset consists of 900 training images and 379 test images for tasks such as lesion segmentation, dermoscopic feature detection, and melanoma classification. It also has a class imbalance.

In [89]:
print(f'Shape of ISBIISBI2016_ISIC_Part3B: {lab_df.shape}')
print(f"Shpae of PH2 dataset: {df_ph2.shape}")

Shape of ISBIISBI2016_ISIC_Part3B: (900, 2)
Shpae of PH2 dataset: (200, 17)


In [90]:
selected_columns = df_ph2.iloc[:, 2:5] 
for column_name, data_type in selected_columns.items():
    unique_values = df_ph2[column_name].value_counts(dropna=False)
    print(unique_values)

Common Nevus
0    120
1     80
Name: count, dtype: int64
Atypical Nevus
0    120
1     80
Name: count, dtype: int64
Melanoma
0    160
1     40
Name: count, dtype: int64


In [91]:
class_distribution = lab_train["diagnosis"].value_counts().reset_index()
class_distribution.columns = ["Diagnosis", "Count"]

fig = px.bar(class_distribution, x="Diagnosis", y="Count", title="Class Distribution")
fig.show()

In [92]:
class_distribution = lab_test["diagnosis"].value_counts().reset_index()
class_distribution.columns = ["Diagnosis", "Count"]

fig = px.bar(class_distribution, x="Diagnosis", y="Count", title="Class Distribution")
fig.show()

## Aspect Ratios and Image Dimensions

In [93]:
def plot_image_dimensions_and_aspect_ratios(folder_path):
    dfs = []
    for root, _, filenames in os.walk(folder_path):
        for path_image in filenames:
            if path_image.endswith('.bmp'):
                image_path = os.path.abspath(os.path.join(root, path_image))
                with Image.open(image_path) as img:
                    width, height = img.size
                    aspect_ratio = width / height
                    df = pd.DataFrame({'Image': [path_image], 'Width': [width], 'Height': [height], 'Aspect Ratio': [aspect_ratio]})
                    dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)

    fig_width = px.histogram(df, x='Width', title='Image Width Distribution')
    fig_height = px.histogram(df, x='Height', title='Image Height Distribution')
    fig_aspect_ratio = px.histogram(df, x='Aspect Ratio', title='Image Aspect Ratio Distribution')

    return fig_width, fig_height, fig_aspect_ratio

fig_width, fig_height, fig_aspect_ratio = plot_image_dimensions_and_aspect_ratios(IMAGE_PH2)
fig_width.show()
fig_height.show()
fig_aspect_ratio.show()


In [94]:
def plot_image_dimensions_and_aspect_ratios(folder_path):
    dfs = []
    for dirpath, _, filenames in os.walk(folder_path):
        for path_image in filenames:
            image_path = os.path.abspath(os.path.join(dirpath, path_image))
            with Image.open(image_path) as img:
                width, height = img.size
                aspect_ratio = width / height
                df = pd.DataFrame({'Image': [path_image], 'Width': [width], 'Height': [height], 'Aspect Ratio': [aspect_ratio]})
                dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)

    fig_width = px.histogram(df, x='Width', title='Image Width Distribution')
    fig_height = px.histogram(df, x='Height', title='Image Height Distribution')
    fig_aspect_ratio = px.histogram(df, x='Aspect Ratio', title='Image Aspect Ratio Distribution')

    return fig_width, fig_height, fig_aspect_ratio

In [95]:
fig_width, fig_height, fig_aspect_ratio = plot_image_dimensions_and_aspect_ratios(IMAGE_ISIC_TRAIN)
fig_width.show()
fig_height.show()
fig_aspect_ratio.show()

In [96]:
fig_width, fig_height, fig_aspect_ratio = plot_image_dimensions_and_aspect_ratios(IMAGE_ISIC_TEST)
fig_width.show()
fig_height.show()
fig_aspect_ratio.show()

The width in the ISIC dataset has a range from 400-4400 pixels and the height has a range from 500-2900 pixels. The aspect ratio is between 0.5 and 1.5. The PH2 dataset has a width range from 760-780 pixel and the height range from 552-578 pixels. The aspect ratio is 1.3. The aspect ratio is not that important for the classification task, but it is important for the segmentation task.

ISIC dataset contains images with a wider range of sizes than the PH2 dataset. The PH2 dataset has more consistent image dimensions and aspectratios.

## Patterns in Data

As shown earlier there are class imbalnces in both datasets regarding the feature diagnosis. The ISIC dataset has a wide range of image dimensions and while the PH2 is consistent. In the PH2 dataset there are image segmentation which are better than in the ISIC dataset. For example the PH2 dataset identifies regions of interest (ROIs) within skin lesions. Analyzing pixel intensity or color distribution within these ROIs can help in the diagnosis of melanoma. The PH2 dataset has color profiles which are feature engineered from images.

In [97]:
color_columns = df_ph2.iloc[:, 11:].columns
combined_counts = df_ph2[color_columns].sum()

fig = px.bar(combined_counts, x=combined_counts.index, y=combined_counts.values, labels={'x': 'Color Category', 'y': 'Count'})
fig.update_layout(
    title='Combined Distribution of Colors',
    xaxis_title='Color Category',
    yaxis_title='Count'
)
fig.show()

## Compatibility of Datasets

While it is technically possible to use multiple datasets together, the PH2 dataset and the ISIC dataset are not compatible with each other. The PH2 dataset has a different image format than the ISIC dataset. The PH2 dataset has a .bmp format and the ISIC dataset has a .jpg format and .png. The shapes of them are also different, in terms of number of images and number of feature columns. The ISIC has 2 and PH2 17. Also regarding the diagnosis labeling in the ISIC dataset it is between (Benign and Malignant) and the PH2 dataset has three different diagnosis (Common Nevus, Atypical Nevus, Melanoma). Which is important if the machine learning project goal differs. 

In all together they are technically compatible, but not in terms of the labeling and the image format. 

## PH2 dataset Train/Test Split

In [98]:
ph2_source_folder = IMAGE_PH2
ph2_train_folder = f'{DATA_PH2}/Train_Data'
ph2_test_folder = f'{DATA_PH2}/Test_Data'

In [ ]:
def preprocess_ph2_dataset(ph2_dataset):
    binary_columns = ['Pigment Network\n(AT/T)', 'Blue-Whitish Veil\n(A/P)', 'Regression Areas\n(A/P)', 'Streaks\n(A/P)']
    binary_mapping = {'AT': 0, 'T': 1, 'A': 0, 'P': 1}
    ph2_dataset[binary_columns] = ph2_dataset[binary_columns].replace(binary_mapping)
    
    ph2_dataset = pd.get_dummies(ph2_dataset, columns=['Dots/Globules\n(A/AT/T)', 'Histological Diagnosis'], 
                                  prefix=['Dots/Globules', 'Histological Diagnosis'], dtype=int)
    
    ph2_dataset.drop(columns=['Histological Diagnosis_0'], inplace=True)
    target = ['Common Nevus', 'Atypical Nevus', 'Melanoma']
    ph2_dataset['diagnosis'] = ph2_dataset[target].idxmax(axis=1)
    ph2_dataset.drop(target, axis=1, inplace=True)
    
    return ph2_dataset

def split_ph2_dataset(ph2_dataset, test_size=0.2, random_state=42):
    y = ph2_dataset['diagnosis']
    X = ph2_dataset.drop(columns='diagnosis', axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    
    return X_train, X_test, y_train, y_test

def move_images_to_folders(split_df, source_folder, destination_folder):
    if os.path.exists(destination_folder):
        shutil.rmtree(destination_folder)
        print(f'Deleted contents of {destination_folder}')
    
    for _, row in split_df.iterrows():
        image_name = row['Image Name']
        source_image_path = os.path.join(source_folder, image_name, image_name + '_Dermoscopic_Image', image_name + '.bmp')
        source_lesion_path = os.path.join(source_folder, image_name, image_name + '_lesion', image_name + '_lesion.bmp')

        if os.path.isfile(source_image_path):
            if not os.path.exists(destination_folder):
                os.makedirs(destination_folder)

            new_image_name = os.path.join(destination_folder, image_name + ".jpg")
            new_lesion_name = os.path.join(destination_folder, image_name + "_Segmentation.png")
            shutil.copy(source_image_path, new_image_name)
            shutil.copy(source_lesion_path, new_lesion_name)
            print(f'Moved {image_name} to {destination_folder}')
        else:
            print(f'Image {image_name} not found in {source_folder}')

ph2_dataset = preprocess_ph2_dataset(df_ph2)

X_train, X_test, y_train, y_test = split_ph2_dataset(ph2_dataset)

move_images_to_folders(X_train, ph2_source_folder, ph2_train_folder)
move_images_to_folders(X_test, ph2_source_folder, ph2_test_folder)

pd.concat([X_train['Image Name'], y_train], axis=1).to_csv(f'{DATA_PH2}/Train_GroundTruth.csv', index=False, header=None)
pd.concat([X_test['Image Name'], y_test], axis=1).to_csv(f'{DATA_PH2}/Test_GroundTruth.csv', index=False, header=None)

# `Task 2: Prepare Keras/PyTorch Data Loaders for PH2 Dataset`

This part has many similarities with the skin_cancer_classification.ipynb. I have used the same code and changed it to fit the PH2 dataset. Thanks for providing the code and documentation.

In [100]:
SEED = 42
BATCH_SIZE = 16
BASE_IMAGE_SIZE = 384, 384
VALID_SIZE = 0.2

##  Data Loading

In [101]:
lab_train = load_data(f'{DATA_PH2}/Train_GroundTruth.csv')
lab_test = load_data(f'{DATA_PH2}/Test_GroundTruth.csv')

In [102]:
def restructure_dataset(source_dir, destination_dir, lab_df):
  labels = dict(zip(lab_df['imageID'], lab_df['diagnosis']))
  destination_dir = Path(destination_dir)
  # create category directories
  for image_path in Path(source_dir).glob("*.jpg"):
    category = labels[image_path.stem]
    category_dir = destination_dir / category
    category_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(image_path, category_dir / image_path.name)

restructure_dataset(
    f'{DATA_PH2}/Train_Data',
    f'{DATA_PH2}/train',
    lab_train
    )

restructure_dataset(
    f'{DATA_PH2}/Test_Data',
    f'{DATA_PH2}/test',
    lab_test
    )

In [103]:
def build_dataset(data_dir, split_val=False, val_size=VALID_SIZE, seed=SEED,
                  shuffle=True, image_size=BASE_IMAGE_SIZE, batch_size=BATCH_SIZE):
  train = tf.keras.utils.image_dataset_from_directory(
      data_dir,
      validation_split=val_size if split_val else None,
      subset="training" if split_val else None,
      seed=seed,
      shuffle=shuffle,
      image_size=image_size,
      batch_size=batch_size)

  if split_val: # Split the dataset into training and validation sets
    val = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=val_size,
        subset="validation",
        seed=seed,
        shuffle=shuffle,
        image_size=image_size,
        batch_size=batch_size)
  else:
    val = None
  return train, val

train_ds, val_ds = build_dataset(f'{DATA_PH2}/train', split_val=True)
test_ds, _ = build_dataset(f'{DATA_PH2}/test', split_val=False, shuffle=False)

assert CLASS_NAMES == train_ds.class_names == val_ds.class_names == test_ds.class_names

Found 160 files belonging to 3 classes.
Using 128 files for training.
Found 160 files belonging to 3 classes.
Using 32 files for validation.
Found 40 files belonging to 3 classes.


## Data Preprocessing and Augmentation

In [104]:
DO_DATA_AUGM = True

def get_preprocessing_with_augm(rescale01, with_augm=DO_DATA_AUGM):
  if rescale01: # range [0,1]
    normalization = tf.keras.layers.Rescaling(scale=1./255)
  else: # range [-1,1]
    normalization = tf.keras.layers.Rescaling(scale=1./127.5, offset=-1)
  preprocessing = tf.keras.Sequential([normalization])
  if with_augm:
    data_augm = tf.keras.Sequential([])
    data_augm.add(tf.keras.layers.RandomRotation(45))
    data_augm.add(tf.keras.layers.RandomTranslation(0.1, 0.1))
    data_augm.add(tf.keras.layers.RandomZoom(0.1, 0.1))
    data_augm.add(tf.keras.layers.RandomFlip(mode="horizontal"))
    data_augm.add(tf.keras.layers.RandomContrast(0.1))
    preprocessing.add(data_augm)
  return preprocessing

def dataset_preprocess(train, val, test, rescale01=True):
  train_preprocess = get_preprocessing_with_augm(rescale01)
  tfm_train = train.map(lambda images, labels: (train_preprocess(images), labels))
  val_preprocess = get_preprocessing_with_augm(rescale01, with_augm=False)
  tfm_val = val.map(lambda images, labels: (val_preprocess(images), labels))
  tfm_test = test.map(lambda images, labels: (val_preprocess(images), labels))
  return tfm_train, tfm_val, tfm_test

tfm_train, tfm_val, tfm_test = dataset_preprocess(train_ds, val_ds, test_ds)

## Visualize the batches

In [105]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [106]:
def show_image_grid(images, titles, n=3):
  if images.min() < 0: # rescaled [-1,1]
    images = (images + 1)/2
  plt.figure(figsize=(10, 10))
  for i, image, title in zip(range(n*n), images, titles):
    ax = plt.subplot(n, n, i + 1)
    ax.imshow(image)
    ax.set_title(title)
    ax.set_axis_off()

def show_samples_from_dataset(dataset):
  images, labels = next(iter(dataset))
  show_image_grid(images.numpy(), [CLASS_NAMES[li] for li in labels])

def dataset_browser_widget(show_samples):
  dataset_widget = widgets.ToggleButtons(
      options=[('Train', tfm_train), ('Valid', tfm_val), ('Test', tfm_test)],
      value=tfm_train,
      description='Dataset:'
  )
  out = widgets.interactive(show_samples, dataset=dataset_widget)
  button_widget = widgets.Button(description="New batch")
  button_widget.on_click(out.update)
  display(widgets.VBox([out, button_widget]))

dataset_browser_widget(show_samples_from_dataset)